In [ ]:
import pandas as pd

df = pd.read_csv("./전국일반음식점.csv", encoding="CP949")   # 파일 경로 알아서 넣어
print(df.columns)

In [ ]:
import pandas as pd
import pymysql
from pyproj import Transformer
import numpy as np

# CSV 로드
df = pd.read_csv("./전국일반음식점.csv", encoding="CP949", low_memory=False)

# 좌표 변환기
transformer = Transformer.from_crs("EPSG:5174", "EPSG:4326", always_xy=True)

# 필요한 컬럼만 추출
df = df[
    ['사업장명','도로명전체주소','소재지전체주소','업태구분명',
     '인허가일자','폐업일자','영업상태명',
     '좌표정보x(epsg5174)','좌표정보y(epsg5174)']
].copy()

# 주소 합치기
df['주소'] = df['도로명전체주소'].fillna(df['소재지전체주소'])

# 날짜 정리
def fix_date(v):
    try:
        return pd.to_datetime(v).date()
    except:
        return None

df['인허가일자'] = df['인허가일자'].apply(fix_date)
df['폐업일자'] = df['폐업일자'].apply(fix_date)

# 좌표 변환
def convert_coord(row):
    x, y = row['좌표정보x(epsg5174)'], row['좌표정보y(epsg5174)']
    try:
        if pd.isna(x) or pd.isna(y):
            return pd.Series([None, None])
        lng, lat = transformer.transform(float(x), float(y))
        return pd.Series([lat, lng])
    except:
        return pd.Series([None, None])

df[['lat','lng']] = df.apply(convert_coord, axis=1)

# NaN → None 변환 (MySQL용)
df = df.replace({np.nan: None})

In [ ]:
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='fls0413!',
    db='dashboard',
    charset='utf8'
)
cur = conn.cursor()

sql = """
INSERT INTO restaurants
(`사업장명`, `주소`, `업태구분명`, `인허가일자`, `폐업일자`, `영업상태명`,
 `x좌표`, `y좌표`, lat, lng)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

batch_size = 1000
data = []

for i, row in df.iterrows():

    # 주소 선택 (둘 중 하나)
    addr = row['도로명전체주소'] if pd.notna(row['도로명전체주소']) else row['소재지전체주소']

    data.append((
        row['사업장명'],
        addr,
        row['업태구분명'],
        row['인허가일자'],
        row['폐업일자'],
        row['영업상태명'],
        row['좌표정보x(epsg5174)'],
        row['좌표정보y(epsg5174)'],
        row['lat'],
        row['lng']
    ))

    if len(data) == batch_size:
        cur.executemany(sql, data)
        conn.commit()
        print(f"{i}행 저장됨")
        data = []

# 잔여 데이터
if data:
    cur.executemany(sql, data)
    conn.commit()

cur.close()
conn.close()

print("완료!")

In [ ]:
import streamlit as st
import pymysql
import pandas as pd

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='fls0413!',
    db='dashboard',
    charset='utf8'
)

df = pd.read_sql("SELECT * FROM restaurants", conn)

st.title("전국 일반음식점 대시보드")

st.map(df[['lat', 'lng']].dropna())